# **01 - ETL & Data Cleaning**

## Objectives

- Load and inspect the raw Online Retail Transaction dataset.
- Assess data quality by identifying missing values, duplicate records, invalid quantities and unit prices.
- Clean and transform the dataset to prepare it for analysis.
- Create a `Revenue` feature to support further analysis.
- Save the cleaned dataset for exploratory data analysis and visualisation.

## Inputs

- Raw Online Retail Transaction dataset stored in the `data` folder.

## Outputs

- Cleaned Online Retail Transaction dataset.
- Cleaned data ready for exploratory data analysis and visualisation.

## Additional Comments

- Data-cleaning decisions will be based on the findings from the initial data inspection and documented throughout the notebook.

---

# Change working directory

In [1]:
import os

current_dir = os.getcwd()
print("Current directory:", current_dir)

if os.path.basename(current_dir) == "jupyter_notebooks":
    os.chdir(os.path.dirname(current_dir))
elif os.path.basename(current_dir) == "vscode-projects":
    os.chdir("Online Retail Transaction Analysis")

current_dir = os.getcwd()
print("Working directory set to:", current_dir)

Current directory: /Users/sahraosman/Documents/vscode-projects/Online Retail Transaction Analysis/jupyter_notebooks
Working directory set to: /Users/sahraosman/Documents/vscode-projects/Online Retail Transaction Analysis


---

# Section 1 -  Import libraries

Here I will be importing the libraries that I will be using in this notebook.

In [2]:
import numpy as np
import pandas as pd


---

# Section 2 - Load the dataset

This section will load the raw Online Retail Transaction dataset and perform an initial inspection to understand its structure.

In [3]:
df = pd.read_csv("data/Online_Retail.csv")
df.head()

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850,United Kingdom


In [4]:
print("Number of rows in the dataset:", df.shape[0])
print("Number of columns in the dataset:", df.shape[1])

Number of rows in the dataset: 541909
Number of columns in the dataset: 8


---

# Section 3 - Initial Data Inspection

Here I'll perform an initial inspection of the dataset to understand its structure, identify any data quality issues, and determine the necessary cleaning steps.

In [5]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 541909 entries, 0 to 541908
Data columns (total 8 columns):
 #   Column       Non-Null Count   Dtype  
---  ------       --------------   -----  
 0   InvoiceNo    541909 non-null  str    
 1   StockCode    541909 non-null  str    
 2   Description  540455 non-null  str    
 3   Quantity     541909 non-null  int64  
 4   InvoiceDate  541909 non-null  str    
 5   UnitPrice    541909 non-null  float64
 6   CustomerID   541909 non-null  int64  
 7   Country      541909 non-null  str    
dtypes: float64(1), int64(2), str(5)
memory usage: 33.1 MB


Looking at the structure of the dataset, we can see that it contains 8 columns and 541,909 rows. The columns are as follows:
`InvoiceNo`, `StockCode`, `Description`, `Quantity`, `InvoiceDate`, `UnitPrice`, `CustomerID`, and `Country`.

I can already see that there are some missing values in the `Description` column. I will need to investigate further to determine the extent of the missing values and decide on an appropriate strategy for handling them.

`InvoiceDate` is currently in string format, and I will need to convert it to a datetime format for analysis.

In [6]:
df.describe(include="all")

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
count,541909,541909,540455,541909.000000,541909,541909.000000,541909.000000,541909
unique,25900,4070,4223,NaN,23260,NaN,NaN,38
top,573585,85123A,WHITE HANGING HEART T-LIGHT HOLDER,NaN,2011-10-31 14:41:00,NaN,NaN,United Kingdom
freq,1114,2313,2369,NaN,1114,NaN,NaN,495478
mean,NaN,NaN,NaN,9.552250,NaN,4.611114,15287.518434,NaN
std,NaN,NaN,NaN,218.081158,NaN,96.759853,1484.746041,NaN
min,NaN,NaN,NaN,-80995.000000,NaN,-11062.060000,12346.000000,NaN
25%,NaN,NaN,NaN,1.000000,NaN,1.250000,14367.000000,NaN
50%,NaN,NaN,NaN,3.000000,NaN,2.080000,15287.000000,NaN
75%,NaN,NaN,NaN,10.000000,NaN,4.130000,16255.000000,NaN


The descriptive statistics provide an overview of both numerical and categorical variables. The dataset contains 25,900 unique invoices, 4,070 unique stock codes and transactions across 38 countries. The United Kingdom accounts for the majority of transaction records.

The `Quantity` and `UnitPrice` columns contain negative minimum values, with `Quantity` ranging from -80,995 to 80,995 and `UnitPrice` ranging from -11,062.06 to 38,970. These values require further investigation during data cleaning.

The `Description` count is lower than the total number of rows, confirming the presence of missing values. Further analysis will be performed to quantify and investigate these records.

---

# Section 4 - Missing Values

In this section, I will investigate missing values within the dataset to determine their extent and decide on an appropriate strategy for handling them.

In [7]:
df.isnull().sum()

InvoiceNo         0
StockCode         0
Description    1454
Quantity          0
InvoiceDate       0
UnitPrice         0
CustomerID        0
Country           0
dtype: int64

The `Description` column contains 1,454 missing values. I will calculate the percentage of missing values and investigate these records further before deciding how to handle them.

In [8]:
df.isnull().sum() / df.shape[0] * 100

InvoiceNo      0.000000
StockCode      0.000000
Description    0.268311
Quantity       0.000000
InvoiceDate    0.000000
UnitPrice      0.000000
CustomerID     0.000000
Country        0.000000
dtype: float64

I can see that the missing values in the `Description` column represent approximately 0.27% of the total dataset, which is a small percentage. I will need to decide whether to drop these rows or impute the missing values based on the context of the analysis.

In [9]:
df[df["Description"].isnull()].head(20)

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
622,536414,22139,NaN,56,2010-12-01 11:52:00,0.0,15287,United Kingdom
1970,536545,21134,NaN,1,2010-12-01 14:32:00,0.0,15287,United Kingdom
1971,536546,22145,NaN,1,2010-12-01 14:33:00,0.0,15287,United Kingdom
1972,536547,37509,NaN,1,2010-12-01 14:33:00,0.0,15287,United Kingdom
1987,536549,85226A,NaN,1,2010-12-01 14:34:00,0.0,15287,United Kingdom
1988,536550,85044,NaN,1,2010-12-01 14:34:00,0.0,15287,United Kingdom
2024,536552,20950,NaN,1,2010-12-01 14:34:00,0.0,15287,United Kingdom
2025,536553,37461,NaN,3,2010-12-01 14:35:00,0.0,15287,United Kingdom
2026,536554,84670,NaN,23,2010-12-01 14:35:00,0.0,15287,United Kingdom
2406,536589,21777,NaN,-10,2010-12-01 16:50:00,0.0,15287,United Kingdom


The initial sample shows that records with a missing `Description` also have a `UnitPrice` of 0. I will investigate whether this pattern applies to all records with a missing description before deciding how to handle them.

In [10]:
df[df["Description"].isnull()]["UnitPrice"].value_counts()

UnitPrice
0.0    1454
Name: count, dtype: int64

All records with a missing `Description` also have a `UnitPrice` of 0. This confirms that the pattern identified in the initial sample applies to all 1,454 missing-description records. I will investigate the wider zero-priced records before deciding how these observations should be handled.

In [11]:
df[df["UnitPrice"] == 0].shape[0]

2515

There are 2,515 records with a `UnitPrice` of 0. Since only 1,454 of these records have a missing `Description`, there are additional zero-priced records that require further investigation.

In [12]:
df[(df["UnitPrice"] == 0) & (df["Description"].notnull())].head(20)

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
6391,536941,22734,amazon,20,2010-12-03 12:08:00,0.0,15287,United Kingdom
6392,536942,22139,amazon,15,2010-12-03 12:08:00,0.0,15287,United Kingdom
7313,537032,21275,?,-30,2010-12-03 16:50:00,0.0,15287,United Kingdom
9302,537197,22841,ROUND CAKE TIN VINTAGE GREEN,1,2010-12-05 14:02:00,0.0,12647,Germany
13217,537425,84968F,check,-20,2010-12-06 15:35:00,0.0,15287,United Kingdom
13218,537426,84968E,check,-35,2010-12-06 15:36:00,0.0,15287,United Kingdom
13264,537432,35833G,damages,-43,2010-12-06 16:10:00,0.0,15287,United Kingdom
14335,537534,85064,CREAM SWEETHEART LETTER RACK,1,2010-12-07 11:48:00,0.0,15287,United Kingdom
14336,537534,84832,ZINC WILLIE WINKIE CANDLE STICK,1,2010-12-07 11:48:00,0.0,15287,United Kingdom
14337,537534,84692,BOX OF 24 COCKTAIL PARASOLS,2,2010-12-07 11:48:00,0.0,15287,United Kingdom


The additional zero-priced records contain a mixture of product descriptions and non-standard descriptions such as `check`, `damages`, and `?`. Some also contain negative quantities. These records will be investigated further during the Quantity and UnitPrice validation stage.

### Missing Value Investigation

The `Description` column contains 1,454 missing values, representing approximately 0.27% of the dataset. Further investigation showed that all 1,454 records with a missing product description also have a `UnitPrice` of 0.

Since these records do not contain a product description and have no recorded unit price, they provide limited value for the product and revenue analysis required for this project. Therefore, these records will be removed from the cleaned dataset.

Further investigation also identified additional records with a `UnitPrice` of 0 that contain valid descriptions. These records will be investigated separately during the validation of `UnitPrice` rather than being removed solely on the basis of the missing-value analysis.

In [13]:
df = df.dropna(subset=["Description"])

In [14]:
df.isnull().sum()

InvoiceNo      0
StockCode      0
Description    0
Quantity       0
InvoiceDate    0
UnitPrice      0
CustomerID     0
Country        0
dtype: int64

After dropping the records with missing `Description` values, we can see that there are no longer any missing values in the dataset.

In [15]:
df.shape[0]

540455

Now there are 540,455 records remaining in the dataset after dropping the records with missing `Description` values. We started with 541,909 rows.

---

# Section 5 - Duplicate Records

In this section, I will investigate duplicate records within the dataset to determine their extent and decide on an appropriate strategy for handling them.

In [16]:
df.duplicated().sum()

np.int64(5268)

Pandas has found 5268 duplicate records in the dataset. I will investigate these records further to determine whether they should be removed or retained based on the context of the analysis.

In [17]:
df[df.duplicated(keep=False)].head(20)

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
485,536409,22111,SCOTTIE DOG HOT WATER BOTTLE,1,2010-12-01 11:45:00,4.95,17908,United Kingdom
489,536409,22866,HAND WARMER SCOTTY DOG DESIGN,1,2010-12-01 11:45:00,2.10,17908,United Kingdom
494,536409,21866,UNION JACK FLAG LUGGAGE TAG,1,2010-12-01 11:45:00,1.25,17908,United Kingdom
517,536409,21866,UNION JACK FLAG LUGGAGE TAG,1,2010-12-01 11:45:00,1.25,17908,United Kingdom
521,536409,22900,SET 2 TEA TOWELS I LOVE LONDON,1,2010-12-01 11:45:00,2.95,17908,United Kingdom
527,536409,22866,HAND WARMER SCOTTY DOG DESIGN,1,2010-12-01 11:45:00,2.10,17908,United Kingdom
537,536409,22900,SET 2 TEA TOWELS I LOVE LONDON,1,2010-12-01 11:45:00,2.95,17908,United Kingdom
539,536409,22111,SCOTTIE DOG HOT WATER BOTTLE,1,2010-12-01 11:45:00,4.95,17908,United Kingdom
548,536412,22327,ROUND SNACK BOXES SET OF 4 SKULLS,1,2010-12-01 11:49:00,2.95,17920,United Kingdom
555,536412,22327,ROUND SNACK BOXES SET OF 4 SKULLS,1,2010-12-01 11:49:00,2.95,17920,United Kingdom


In [18]:
duplicates = df[df.duplicated(keep=False)]

duplicates.sort_values(
    by=["InvoiceNo", "StockCode", "Quantity", "UnitPrice"]
).head(30)

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
494,536409,21866,UNION JACK FLAG LUGGAGE TAG,1,2010-12-01 11:45:00,1.25,17908,United Kingdom
517,536409,21866,UNION JACK FLAG LUGGAGE TAG,1,2010-12-01 11:45:00,1.25,17908,United Kingdom
485,536409,22111,SCOTTIE DOG HOT WATER BOTTLE,1,2010-12-01 11:45:00,4.95,17908,United Kingdom
539,536409,22111,SCOTTIE DOG HOT WATER BOTTLE,1,2010-12-01 11:45:00,4.95,17908,United Kingdom
489,536409,22866,HAND WARMER SCOTTY DOG DESIGN,1,2010-12-01 11:45:00,2.10,17908,United Kingdom
527,536409,22866,HAND WARMER SCOTTY DOG DESIGN,1,2010-12-01 11:45:00,2.10,17908,United Kingdom
521,536409,22900,SET 2 TEA TOWELS I LOVE LONDON,1,2010-12-01 11:45:00,2.95,17908,United Kingdom
537,536409,22900,SET 2 TEA TOWELS I LOVE LONDON,1,2010-12-01 11:45:00,2.95,17908,United Kingdom
578,536412,21448,12 DAISY PEGS IN WOOD BOX,1,2010-12-01 11:49:00,1.65,17920,United Kingdom
598,536412,21448,12 DAISY PEGS IN WOOD BOX,1,2010-12-01 11:49:00,1.65,17920,United Kingdom


The duplicate inspection confirms that some records are exact copies, with identical values across all columns including `InvoiceNo`, `StockCode`, `Description`, `Quantity`, `InvoiceDate`, `UnitPrice`, `CustomerID`, and `Country`.

Repeated customers, products or invoices are expected in retail transaction data and are not considered duplicates unless the entire row is identical. Exact duplicate records could cause quantities and revenue to be counted more than once, so these records will be removed before analysis.

In [19]:
df = df.drop_duplicates(keep='first')

This keeps the first occurrence of each duplicate record and removes subsequent identical occurrences.

In [20]:
df.duplicated().sum()

np.int64(0)

We can see now that after removing the duplicate records, there are no longer any duplicate records in the dataset.

In [21]:
df.shape[0]

535187

After removing 5,268 exact duplicate records, 535,187 records remain in the dataset.

### Duplicate Records Conclusion

The dataset contained 5,268 exact duplicate records. Inspection confirmed that these records had identical values across all columns.

Repeated customers, products and invoices are expected within retail transaction data and were not considered duplicates unless the entire record was identical. Exact duplicates were removed while keeping the first occurrence to prevent potential double-counting of quantities and revenue.

After removing the duplicates, no exact duplicate records remain in the dataset.

# Conclusions & Next Steps